# **Análisis Exploratorio de Datos: Hotel Bookings**
## Análisis estadístico riguroso de patrones de reserva hotelera con PySpark
### Michael Morantes
#### Machine Learning con Pyspark y docker
#### 17/03/2026
##### Bogota, Colombia
---

# 1. Introducción y Contexto

## Descripción del Dataset

Este dataset contiene **119,390 registros de reservas hoteleras** provenientes de dos propiedades: un hotel resort vacacional y un hotel urbano/ciudad. Los datos abarcan desde el **año 2015 hasta 2017** y representan un ciclo completo de operaciones hoteleras.

**Fuente**: Dataset de Kaggle - Hotel Bookings

**Variables disponibles**:
- 20 variables numéricas: comportamiento de booking (lead_time, estancias), características del huésped (adultos, niños, bebés), indicadores de comportamiento previo, demanda y precios (ADR)
- 12 variables categóricas: tipo de hotel, canal de distribución, segmento de mercado, país de origen, tipo de habitación, estado de reserva

**Contexto del problema**:

En la industria hotelera, la gestión de reservas y la predicción de cancelaciones son críticas para:
1. **Optimización de ingresos (Revenue Management)**: Evitar overbooking vs. subutilización de capacidad
2. **Planificación operacional**: Cantidad de personal, inventario de alimentos, housekeeping
3. **Estrategia comercial**: Identificar segmentos rentables, canales efectivos, patrones estacionales

Con una **tasa de cancelación del ~37%**, este dataset presenta un problema real y complejo.

## Justificación Estadística del Análisis

Este análisis es relevante porque:

1. **Problema de desequilibrio de clases**: ~37% de cancelaciones vs ~63% de check-outs genera complejidad estadística. Las medidas de tendencia central pueden ser engañosas.

2. **Multicolinealidad esperada**: Múltiples variables temporales (lead_time, arrival_date_*) y comportamentales pueden estar correlacionadas. Es fundamental detectar esto.

3. **Sesgo de distribución**: Lead_time muestra valores extremos (mín=0, máx=737 días). La presencia de outliers requiere análisis específico.

4. **Heterogeneidad de mercados**: Dos tipos de hotel (resort vs. ciudad) operan bajo dinámicas completamente diferentes.

5. **Datos faltantes estructurados**: 94% de datos faltantes en 'company' no son aleatorios—reflejan canales de distribución específicos.

## Preguntas de Investigación

### **Pregunta 1: ¿Cuáles son los perfiles de comportamiento de booking distintos y cómo se relacionan con la cancelación?**

**Especificidad**: Más allá de solo cancelación, nos interesa entender si existen patrones latentes en el comportamiento de reserva (lead_time, cambios en booking, solicitudes especiales) que caractericen grupos de huéspedes.

**Medibilidad**: Segmentar por quartiles de lead_time, número de cambios en booking, y solicitudes especiales. Calcular tasas de cancelación dentro de cada segmento.

### **Pregunta 2: ¿Existe una relación significativa entre el canal de distribución/segmento de mercado y la probabilidad de cancelación?**

**Especificidad**: No solo queremos saber si hay diferencias, sino si la magnitud y dirección de esas diferencias cambian según el tipo de hotel.

**Medibilidad**: Tablas de contingencia, tasas de cancelación por canal × tipo hotel, análisis de interacción.

### **Pregunta 3: ¿Cuál es la relación entre duración de estancia, composición del grupo (adultos/niños), y el ADR?**

**Especificidad**: Las variables no funcionan independientemente. Un grupo grande de niños en un fin de semana extenso probablemente genera dinámicas muy diferentes.

**Medibilidad**: Correlaciones, análisis de segmentación, tasas de cancelación por categoría.

## Hipótesis Preliminares

### **H1: Comportamiento de Booking**
- **Hipótesis alternativa**: Reservas con lead_time alto (>60 días) + cambios en booking (>0) + solicitudes especiales (>0) tienen menor tasa de cancelación (~25%) que booking impulsivo (<25 días, sin cambios, ~45% cancelación).
- **Base teórica**: Inversión de tiempo/atención señala compromiso. Cambios post-reserva reflejan anticipación del viaje.

### **H2: Efecto del Canal de Distribución**
- **Hipótesis alternativa**: Direct bookings tienen menor cancelación (~30%) que intermediarios (TA/TO, ~40%+) porque generan menor barrera de salida.
- **Base teórica**: Irreversibilidad percibida (dinero transaccional directo) vs. virtual (a través de terceros).

### **H3: Duración + Composición de Grupo → Cancelación**
- **Hipótesis alternativa**: Se espera que la duración de la estancia y la composición del grupo influyan significativamente en el ADR, donde estancias más largas tienden a asociarse con un menor precio promedio por noche, mientras que la presencia de niños y el tamaño del grupo pueden modificar dicha relación.
- **Base teórica**:El ADR se ve influenciado por estrategias de precios, donde estancias más largas reducen el costo por noche, mientras que la composición del grupo modifica la demanda y el tipo de habitación, generando variaciones en el precio.

---

# 2. Carga y Exploración Inicial

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, isnan, isnull, count, mean, stddev, 
    min as spark_min, max as spark_max, approx_percentile,
    sum as spark_sum, avg, countDistinct, coalesce,
    skewness, kurtosis
)
import pandas as pd
import numpy as np

# Crear sesión de Spark
spark = SparkSession.builder \
    .appName("Hotel_Bookings_EDA") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✓ Sesión Spark inicializada correctamente")

✓ Sesión Spark inicializada correctamente


In [14]:
# Lectura del dataset con PySpark
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("nullValue", "NULL") \
    .csv("/home/jovyan/work/data/hotel_bookings.csv")

print(f"✓ Dataset cargado: {df.count()} filas")

✓ Dataset cargado: 119390 filas


## 2.1 Esquema del Dataset

In [15]:
df.printSchema()

root
 |-- hotel: string (nullable = true)
 |-- is_canceled: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_date_year: integer (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: integer (nullable = true)
 |-- arrival_date_day_of_month: integer (nullable = true)
 |-- stays_in_weekend_nights: integer (nullable = true)
 |-- stays_in_week_nights: integer (nullable = true)
 |-- adults: integer (nullable = true)
 |-- children: string (nullable = true)
 |-- babies: integer (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_type: string (nullab

**Interpretación del esquema**:

- **Variables temporales** (IntegerType): arrival_date_year, arrival_date_week_number, lead_time. Permiten análisis de patrones estacionales.
- **Variables de composición** (IntegerType/DoubleType): adults, children, babies. Nota: children es DoubleType por registros incompletos.
- **Variables de duración** (IntegerType): stays_in_weekend_nights, stays_in_week_nights.
- **Variables de comportamiento** (IntegerType): booking_changes, total_of_special_requests, days_in_waiting_list.
- **Variables de pricing** (DoubleType): adr (Average Daily Rate).
- **Variables categóricas** (StringType): hotel, country, market_segment, distribution_channel, etc.

## 2.2 Dimensiones del Dataset

In [16]:
n_rows = df.count()
n_cols = len(df.columns)

print(f"Dimensiones del dataset: {n_rows:,} filas × {n_cols} columnas")
print(f"Período de datos: 2015-2017 (3 años de operaciones)")
print(f"Dos tipos de propiedad: Resort Hotel + City Hotel")

Dimensiones del dataset: 119,390 filas × 32 columnas
Período de datos: 2015-2017 (3 años de operaciones)
Dos tipos de propiedad: Resort Hotel + City Hotel


## 2.3 Análisis de Valores Nulos

In [17]:
# Análisis de valores nulos
null_counts = df.select([
    count(when(isnull(col(c)), 1)).alias(c)
    for c in df.columns
]).collect()[0]

null_analysis = []
for col_name in df.columns:
    null_count = getattr(null_counts, col_name)
    null_pct = (null_count / n_rows * 100)
    if null_pct > 0:
        null_analysis.append({
            'Columna': col_name,
            'Nulos': null_count,
            'Porcentaje': round(null_pct, 2)
        })

null_df = pd.DataFrame(null_analysis).sort_values('Porcentaje', ascending=False)
print(null_df.to_string(index=False))
print(f"\nTotal de columnas con valores faltantes: {len(null_df)}")

Columna  Nulos  Porcentaje
company 112593       94.31
  agent  16340       13.69
country    488        0.41

Total de columnas con valores faltantes: 3


**Análisis e Interpretación**:

- **company (94.31% nulos)**: Casi completamente nula. Prácticamente no informativa. **Decisión**: Excluir del análisis.
- **agent (13.69% nulos)**: Ausente en reservas directas. **Decisión**: Mantener pero crear variable binaria.
- **country (0.41% nulos)**: Mínimo. **Decisión**: Imputar como "Unknown".
- **children**: 4 valores NULL. **Decisión**: Convertir a IntegerType, imputar como 0.

In [18]:
from pyspark.sql.functions import lit

# Convertir children a int, imputar nulos a 0
df = df.withColumn(
    "children",
    coalesce(col("children").cast("int"), lit(0))  # ✓ CORRECTO
)

# Crear variables binarias
df = df.withColumn(
    "has_agent",
    when(col("agent").isNull(), lit(0)).otherwise(lit(1))  # ✓ CORRECTO
)

df = df.withColumn(
    "has_company",
    when(col("company").isNull(), lit(0)).otherwise(lit(1))  # ✓ CORRECTO
)

# Excluir company
df = df.drop("company")

# Imputar country nulos
df = df.withColumn(
    "country",
    coalesce(col("country"), lit("Unknown"))  # ✓ CORRECTO
)

print("✓ Limpieza de datos completada")

✓ Limpieza de datos completada


In [19]:
# Crear variables derivadas para análisis
df = df.withColumn(
    "total_stay_nights",
    col("stays_in_weekend_nights") + col("stays_in_week_nights")
)

df = df.withColumn(
    "total_guests",
    col("adults") + col("children") + col("babies")
)

print("✓ Variables derivadas creadas:")
print("  - total_stay_nights")
print("  - total_guests")

✓ Variables derivadas creadas:
  - total_stay_nights
  - total_guests


---

# 3. Estadística Descriptiva

## 3.1 Variables Numéricas

In [25]:
from pyspark.sql.functions import col, mean, stddev, min as spark_min, max as spark_max, expr
import pandas as pd

# Definir variables numéricas
numeric_vars = [
    'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
    'adults', 'children', 'babies', 'adr', 'booking_changes',
    'total_of_special_requests', 'previous_cancellations', 'total_guests'
]

# Calcular estadísticas para TODAS las variables
stats_list = []

for var in numeric_vars:
    stats = df.select(
        mean(col(var)).alias('mean'),
        expr(f'percentile_approx({var}, 0.5)').alias('median'),
        stddev(col(var)).alias('std'),
        spark_min(col(var)).alias('min'),
        spark_max(col(var)).alias('max')
    ).collect()[0]
    
    stats_list.append({
        'Variable': var,
        'Mean': round(stats['mean'] or 0, 2),
        'Median': round(stats['median'] or 0, 2),
        'Std': round(stats['std'] or 0, 2),
        'Min': round(stats['min'] or 0, 2),
        'Max': round(stats['max'] or 0, 2)
    })

# Convertir a DataFrame de pandas y mostrar completo
stats_df = pd.DataFrame(stats_list)
print(stats_df.to_string(index=False))

                 Variable   Mean  Median    Std   Min    Max
                lead_time 104.01    69.0 106.86  0.00  737.0
  stays_in_weekend_nights   0.93     1.0   1.00  0.00   19.0
     stays_in_week_nights   2.50     2.0   1.91  0.00   50.0
                   adults   1.86     2.0   0.58  0.00   55.0
                 children   0.10     0.0   0.40  0.00   10.0
                   babies   0.01     0.0   0.10  0.00   10.0
                      adr 101.83    94.5  50.54 -6.38 5400.0
          booking_changes   0.22     0.0   0.65  0.00   21.0
total_of_special_requests   0.57     0.0   0.79  0.00    5.0
   previous_cancellations   0.09     0.0   0.84  0.00   26.0
             total_guests   1.97     2.0   0.72  0.00   55.0


### Interpretación de variables numéricas críticas

- **lead_time**: Presenta una fuerte asimetría positiva (media 104 > mediana 69), lo que indica que la mayoría de reservas se realizan con anticipación moderada, pero existen casos extremos de planificación muy anticipada (hasta 737 días).

- **stays_in_weekend_nights / stays_in_week_nights**: La mediana baja (1 y 2 noches) confirma que predominan estancias cortas. Los valores máximos elevados evidencian outliers asociados a estancias prolongadas poco frecuentes.

- **adults**: La media cercana a 2 y mediana igual a 2 indica que el perfil dominante es de parejas. Los valores máximos son atípicos y poco representativos.

- **children y babies**: Ambas variables tienen medianas de 0 y medias muy bajas, lo que indica que la gran mayoría de reservas no incluyen niños ni bebés, caracterizando un mercado principalmente adulto.

- **adr**: Existe asimetría positiva (media 101.83 > mediana 94.5) y presencia de outliers extremos (máximo 5400). Además, el valor mínimo negativo sugiere posibles anomalías o ajustes en los datos.

- **booking_changes**: La mediana de 0 indica que la mayoría de reservas no presentan modificaciones, aunque algunos casos muestran múltiples cambios (hasta 21), lo que refleja comportamientos más activos de ciertos clientes.

- **total_of_special_requests**: La mayoría de clientes no realiza solicitudes especiales (mediana 0), pero existe una minoría que sí interactúa más con el servicio.

- **previous_cancellations**: La mediana de 0 indica que la mayoría de clientes no tiene historial de cancelaciones, aunque hay casos aislados con múltiples cancelaciones previas.

- **total_guests**: La media cercana a 2 y mediana de 2 confirma que predominan grupos pequeños, principalmente parejas. Los valores máximos son outliers.

In [27]:
# Skewness y Kurtosis
skew_list = []
for var in numeric_vars[:5]:
    stats = df.select(
        skewness(col(var)).alias('skewness'),
        kurtosis(col(var)).alias('kurtosis')
    ).collect()[0]
    
    skew_list.append({
        'Variable': var,
        'Skewness': round(stats['skewness'] or 0, 3),
        'Kurtosis': round(stats['kurtosis'] or 0, 3)
    })

skew_df = pd.DataFrame(skew_list)
print("ASIMETRÍA (SKEWNESS) Y CURTOSIS\n")
print(skew_df.to_string(index=False))
print("\nInterpretación: Skewness > 0 = cola derecha, |Skewness| > 1 = muy asimétrica")

ASIMETRÍA (SKEWNESS) Y CURTOSIS

               Variable  Skewness  Kurtosis
              lead_time     1.347     1.696
stays_in_weekend_nights     1.380     7.174
   stays_in_week_nights     2.862    24.283
                 adults    18.318  1352.058
               children     4.113    18.674

Interpretación: Skewness > 0 = cola derecha, |Skewness| > 1 = muy asimétrica


## 3.2 Variables Categóricas

In [30]:
# ============================================================================
# 10. VARIABLES CATEGÓRICAS
# ============================================================================

print("\n" + "="*80)
print("ANÁLISIS DE VARIABLES CATEGÓRICAS")
print("="*80)

# Obtener todas las variables categóricas automáticamente
categorical_vars = [col for col in df.columns if df.schema[col].dataType.typeName() == 'string']

print(f"\nTotal de variables categóricas: {len(categorical_vars)}\n")
print("="*80)

# Analizar TODAS las variables categóricas
for cat_var in sorted(categorical_vars):
    freqs = df.groupBy(cat_var).count().collect()
    
    cat_list = []
    for row in freqs:
        cat_list.append({
            cat_var: getattr(row, cat_var),
            'Frecuencia': row['count'],
            'Porcentaje': round(row['count'] / n_rows * 100, 2)
        })
    
    cat_df = pd.DataFrame(cat_list).sort_values('Frecuencia', ascending=False)
    
    print(f"\n{cat_var.upper()} (Cardinalidad: {len(freqs)}):")
    print(cat_df.to_string(index=False))
    print()


ANÁLISIS DE VARIABLES CATEGÓRICAS

Total de variables categóricas: 11


ARRIVAL_DATE_MONTH (Cardinalidad: 12):
arrival_date_month  Frecuencia  Porcentaje
            August       13877       11.62
              July       12661       10.60
               May       11791        9.88
           October       11160        9.35
             April       11089        9.29
              June       10939        9.16
         September       10508        8.80
             March        9794        8.20
          February        8068        6.76
          November        6794        5.69
          December        6780        5.68
           January        5929        4.97


ASSIGNED_ROOM_TYPE (Cardinalidad: 12):
assigned_room_type  Frecuencia  Porcentaje
                 A       74053       62.03
                 D       25322       21.21
                 E        7806        6.54
                 F        3751        3.14
                 G        2553        2.14
                 C        2375

---

### Interpretación de variables categóricas (resumen clave)

- Existe **estacionalidad clara**, con mayor concentración de reservas en meses de verano (julio y agosto).

- El mercado está **altamente concentrado en pocos segmentos**, predominando:
  - Clientes **Transient** (~75%)
  - Canal **TA/TO (agencias)** (~82%)
  - Segmento **Online TA** (~47%)

- La mayoría de reservas son **sin depósito (~88%)**, lo que puede facilitar cancelaciones.

- Predomina el **City Hotel (~66%)**, indicando enfoque urbano.

- Alta concentración en pocos tipos de habitación (A y D).

- Fuerte concentración geográfica en **Portugal (~41%)**.

- Alta tasa de cancelación (~36%), relevante para análisis.

---

# 4. Consultas con Spark SQL

In [18]:
# Registrar como tabla temporal
df.createOrReplaceTempView("hotel_bookings")
print("✓ Tabla 'hotel_bookings' registrada en Spark SQL")

✓ Tabla 'hotel_bookings' registrada en Spark SQL


## Consulta 1: Filtro WHERE - Lead time largo

In [37]:
# Consulta 1: Reservas con lead_time > 180 días
consulta1 = spark.sql("""
    SELECT 
        hotel,
        COUNT(*) as cantidad,
        ROUND(AVG(lead_time), 2) as lead_time_promedio,
        ROUND(SUM(CASE WHEN is_canceled = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as tasa_cancelacion_pct
    FROM hotel_bookings
    WHERE lead_time > 180
    GROUP BY hotel
    ORDER BY cantidad DESC
""")

print("CONSULTA 1: Reservas con lead_time > 180 días\n")
consulta1.show()

CONSULTA 1: Reservas con lead_time > 180 días

+------------+--------+------------------+--------------------+
|       hotel|cantidad|lead_time_promedio|tasa_cancelacion_pct|
+------------+--------+------------------+--------------------+
|  City Hotel|   16948|            286.98|               64.06|
|Resort Hotel|    7744|            255.92|               41.58|
+------------+--------+------------------+--------------------+



**Interpretación:**
- Línea 1: WHERE filtra solo registros donde lead_time > 180
- Línea 2: GROUP BY agrupa por tipo de hotel
- Línea 3: COUNT(*) cuenta total de reservas
- Línea 4: SUM(CASE...) cuenta cancelaciones usando lógica condicional
- **Resultado**: City Hotel domina (~75%), con tasa de cancelación de 42%

## Consulta 2: GROUP BY - Canal de distribución

In [22]:
# Consulta 2: Tasas por canal
consulta2 = spark.sql("""
    SELECT 
        distribution_channel,
        COUNT(*) as total_bookings,
        ROUND(SUM(CASE WHEN is_canceled = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as tasa_cancelacion_pct,
        ROUND(AVG(adr), 2) as adr_promedio
    FROM hotel_bookings
    GROUP BY distribution_channel
    ORDER BY total_bookings DESC
""")

print("CONSULTA 2: Cancelación por canal de distribución\n")
consulta2.show()

CONSULTA 2: Cancelación por canal de distribución

+--------------------+--------------+--------------------+------------+
|distribution_channel|total_bookings|tasa_cancelacion_pct|adr_promedio|
+--------------------+--------------+--------------------+------------+
|               TA/TO|         97870|               41.03|      103.29|
|              Direct|         14645|               17.46|      106.65|
|           Corporate|          6677|               22.08|       69.33|
|                 GDS|           193|               19.17|      120.55|
|           Undefined|             5|               80.00|       46.24|
+--------------------+--------------+--------------------+------------+



**Interpretación:**
- Direct: 17,46% cancelación (menor)
- TA/TO: 41,03% cancelación (mayor)
- Corporate: 22,08% cancelación (muy bajo)
- **Conclusión**: El canal SÍ es factor significativo 

## Consulta 3: Funciones de agregación - Depósito

In [38]:
# Consulta 3: Impacto de depósito
consulta3 = spark.sql("""
    SELECT 
        deposit_type,
        COUNT(*) as total_reservas,
        ROUND(SUM(CASE WHEN is_canceled = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as tasa_cancelacion_pct,
        ROUND(AVG(adr), 2) as adr_promedio
    FROM hotel_bookings
    GROUP BY deposit_type
    ORDER BY tasa_cancelacion_pct ASC
""")

print("CONSULTA 3: Impacto del tipo de depósito\n")
consulta3.show()

CONSULTA 3: Impacto del tipo de depósito

+------------+--------------+--------------------+------------+
|deposit_type|total_reservas|tasa_cancelacion_pct|adr_promedio|
+------------+--------------+--------------------+------------+
|  Refundable|           162|               22.22|       76.21|
|  No Deposit|        104641|               28.38|      103.53|
|  Non Refund|         14587|               99.36|       89.96|
+------------+--------------+--------------------+------------+



---
**Interpretación**

- **Non Refund** presenta una tasa de cancelación extremadamente alta (~99.36%), lo que indica que casi todas estas reservas terminan cancelándose o están mal clasificadas.

- **No Deposit** tiene una tasa de cancelación moderada (~28.38%), siendo el comportamiento más representativo del dataset.

- **Refundable** muestra la menor tasa de cancelación (~22.22%), aunque con muy pocos registros, por lo que su representatividad es limit  a.

- En términos de **ADR**, las reservas sin depósito tienen el valor promedio má alto (~103.53), mientras que las reembolsables presentan menr ADR (~76.21).

- **Conclusión:** El tipo de depósito sí influye en la cancelación, pero los resultados sugieren posibles anomalías en la categoría *Non Refund*, por lo que debe analizarse con cautela.

# 4.5 Correlaciones y Relaciones n cautela.

In [39]:
print("\n" + "="*80)
print("MATRIZ DE CORRELACIÓN")
print("="*80)
 
correlation_vars = [
    'is_canceled', 'lead_time', 'total_stay_nights', 'adults', 'children',
    'adr', 'booking_changes', 'total_of_special_requests',
    'previous_cancellations', 'previous_bookings_not_canceled'
]
 
assembler = VectorAssembler(inputCols=correlation_vars, outputCol="features")
df_assembled = assembler.transform(df)
 
correlation_matrix = Correlation.corr(df_assembled, "features").collect()[0][0].toArray()
 
corr_df = pd.DataFrame(
    correlation_matrix,
    index=correlation_vars,
    columns=correlation_vars
)
 
print("\nMatriz de Correlación Pearson:")
print(corr_df.round(3).to_string())
 


MATRIZ DE CORRELACIÓN

Matriz de Correlación Pearson:
                                is_canceled  lead_time  total_stay_nights  adults  children    adr  booking_changes  total_of_special_requests  previous_cancellations  previous_bookings_not_canceled
is_canceled                           1.000      0.293              0.018   0.060     0.005  0.048           -0.144                     -0.235                   0.110                          -0.057
lead_time                             0.293      1.000              0.157   0.120    -0.038 -0.063            0.000                     -0.096                   0.086                          -0.074
total_stay_nights                     0.018      0.157              1.000   0.105     0.051  0.068            0.096                      0.079                  -0.015                          -0.053
adults                                0.060      0.120              0.105   1.000     0.030  0.231           -0.052                      0.123       

In [41]:
print("\n" + "="*80)
print("TOP CORRELACIONES CON is_canceled")
print("="*80)
 
canceled_corr = corr_df['is_canceled'].drop('is_canceled').abs().sort_values(ascending=False)
 
print("\nTop 5 correlaciones (valor absoluto):")
for i, (var, corr_val) in enumerate(canceled_corr.head(5).items(), 1):
    actual_corr = corr_df.loc['is_canceled', var]
    print(f"{i}. {var:30s}: {actual_corr:7.4f} (r²={corr_val**2:.4f})")


TOP CORRELACIONES CON is_canceled

Top 5 correlaciones (valor absoluto):
1. lead_time                     :  0.2931 (r²=0.0859)
2. total_of_special_requests     : -0.2347 (r²=0.0551)
3. booking_changes               : -0.1444 (r²=0.0208)
4. previous_cancellations        :  0.1101 (r²=0.0121)
5. adults                        :  0.0600 (r²=0.0036)


---
## Interpretación: Correlaciones con cancelación

En general, ninguna variable explica por sí sola la cancelación. Todas las relaciones son **débiles**, lo que significa que cancelar una reserva depende de varios factores al mismo tiempo.

### Lo más importante

- **lead_time (0.29)**  
  Es la variable más relevante. Entre más tiempo de anticipación tiene una reserva, **más probable es que se cancele**.  
  Tiene sentido: planes a largo plazo cambian más.

- **total_of_special_requests (-0.23)**  
  Cuando un cliente hace solicitudes especiales, **es menos probable que cancele**.  
  Esto indica mayor interés o compromiso con el viaje.

- **booking_changes (-0.14)**  
  Si una reserva tiene cambios, **también tiende a cancelarse menos**.  
  Ajustar la reserva muestra intención de mantener el viaje.

- **previous_cancellations (0.11)**  
  Quienes han cancelado antes, **tienden a volver a cancelar**, pero el efecto es pequeño.

- **adults (0.06)**  
  El número de adultos **no influye casi nada** en la cancelación.

---

### En pocas palabras

La cancelación no depende de una sola cosa.  
Pero sí se ve claro que **el comportamiento del cliente (anticipación, cambios, solicitudes)** importa más que características simples como el número de personas.
ecialmente en variables críticas como el tipo de depósito.


---

# 5. Conclusiones


### Respuesta a las preguntas de investigación

**Pregunta 1: Perfiles de booking**

A partir del análisis de reservas con alta anticipación (lead_time > 180 días), se observa que los clientes no necesariamente son más estables. De hecho, los *City Hotel* presentan una tasa de cancelación considerablemente alta (~64%), mientras que los *Resort Hotel* muestran una menor (~41%).

Esto sugiere que un mayor lead_time no siempre implica mayor compromiso, sino que puede estar asociado a mayor incertidumbre en el viaje.

**Hipótesis H1: Parcialmente confirmada.**  
Aunque se esperaba que mayor anticipación redujera cancelaciones, los resultados muestran que depende del tipo de cliente y del contexto (por ejemplo, hoteles urbanos vs vacacionales).

---

**Pregunta 2: Canal de distribución**

Se evidencia una diferencia clara en las tasas de cancelación según el canal:

- **TA/TO** presenta la mayor tasa (~41%)
- **Direct** tiene la menor (~17%)
- **Corporate** y **GDS** muestran valores intermedios (~19%–22%)

Esto indica que los clientes que reservan directamente tienen mayor compromiso, mientras que las reservas a través de intermediarios tienden a ser más volátiles.

**Hipótesis H2: Confirmada.**  
El canal de distribución influye significativamente en la probabili**Pregunta 3: relación entre duración de estancia, composición del grupo, y el ADR**

La hipótesis H3 se cumple de forma parcial.

Por un lado, la duración de la estancia muestra una relación débil con las demás variables, lo que indica que, aunque puede influir en el ADR, su efecto no es lo suficientemente fuerte como para explicarlo por sí sola.

Por otro lado, la composición del grupo (especialmente la presencia de niños) parece tener un papel más relevante en el comportamiento de la reserva. Esto sugiere que el tipo de cliente influye tanto en el precio como en la dinámica de la reserva.

En conjunto, los resultados indican que el ADR no depende únicamente de la duración o del tamaño del grupo de forma aislada, sino de la combinación de estos factores junto con otras variables del contexto de la reserva.

Por lo tanto, aunque la teoría sugiere una relación clara, en los datos esta se presenta de manera más compleja y con efectos moderados.cliente. por lo que requiere análisis adicional.

---

### Limitaciones del análisis

- Los datos corresponden a un periodo previo a la pandemia, por lo qocos registros, lo que limita su interpretación.
- Existen posibles anomalías en variables como *deposit_type*, que pueden afectar las conclusiones.
- El análisis es descriptivo, por lo que no permite establecer relaciones causales.

---

### Propuestas de análisis futuro

- Construir un modelo predictivo de cancelación para identificar variables más influyentes.
- Analizar interacciones entre variables (por ejemplo, canal + lead_time).
- Evaluar el impacto del precio (ADR) en la probabilidad de cancelación.
- Explorar posibles inconsistencias en los datos, especialmente en variables críticas como el tipo de depósito.
